## EX1 - 与 TOES 的对比实验 (Benchmark Q1)

### Cell EX1.1 — Mock ROS

In [ ]:
import sys
from unittest.mock import MagicMock

class MockObject(MagicMock):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

sys.modules["rospy"] = MockObject()
sys.modules["grid_map_msgs"] = MockObject()
sys.modules["grid_map_msgs.msg"] = MockObject()
sys.modules["std_msgs"] = MockObject()
sys.modules["std_msgs.msg"] = MockObject()
sys.modules["geometry_msgs"] = MockObject()
sys.modules["geometry_msgs.msg"] = MockObject()
sys.modules["visualization_msgs"] = MockObject()
sys.modules["visualization_msgs.msg"] = MockObject()
sys.modules["drone_env_viz"] = MockObject()
sys.modules["drone_env_viz.msg"] = MockObject()
sys.modules["tf"] = MockObject()

print("✅ ROS 模块伪装完成。")

✅ ROS 模块已伪装完成 (MagicMock)，可以导入 TOES 库了。


### 定义 Diffusion 模型接口 (Wrapper)

In [8]:
import numpy as np
import time
import torch
# 假设路径已配置好，可以直接 import
from experiments.bias_search.target_distribution import TargetDistribution
from experiments.bias_search.build_solver import build_erg_time_opt_solver

class TOESSolver:
    def __init__(self, workspace_bounds=np.array([[0., 3.5], [-1., 3.5]])):
        self.workspace_bounds = workspace_bounds

    def solve(self, start, goal, distribution_manager, gamma, max_iter=2000):
        x0 = np.array([start[0], start[1], 0., 0.])
        xf = np.array([goal[0], goal[1], 0., 0.])
        args = {
            'N': 100, 'x0': x0, 'xf': xf,
            'erg_ub': gamma, 'alpha': 0.5,
            'wrksp_bnds': self.workspace_bounds
        }
        solver, _ = build_erg_time_opt_solver(args, distribution_manager)
        
        t0 = time.time()
        solver.solve(args=args, max_iter=max_iter, eps=1e-5)
        solve_time = time.time() - t0
        
        sol = solver.get_solution()
        return {'trajectory': sol['x'][:, :2], 'time': solve_time}

class DiffusionSolver:
    def __init__(self, model, device):
        self.model = model
        self.device = device
        rsn = model.config.normalizer.robot_state
        self.rs_mean = torch.as_tensor(rsn.mean, device=device)
        self.rs_std = torch.as_tensor(rsn.std, device=device)

    def solve(self, start, dist_grid, packed, mask, gamma):
        rs_phys = torch.tensor([start[0], start[1], 0., 0.], device=self.device).unsqueeze(0)
        rs_norm = (rs_phys - self.rs_mean) / self.rs_std
        gamma_tensor = torch.tensor([gamma], device=self.device, dtype=torch.float32).view(1, 1)
        
        inputs = {
            "distribution": dist_grid.unsqueeze(0).to(self.device),
            "robot_state": rs_norm,
            "gaussian_packed": packed.unsqueeze(0).to(self.device),
            "gaussian_padding_mask": mask.unsqueeze(0).to(self.device),
            "gamma": gamma_tensor
        }
        
        torch.cuda.synchronize()
        t0 = time.time()
        with torch.no_grad():
            out = self.model.inference(inputs)
        torch.cuda.synchronize()
        solve_time = time.time() - t0
        
        return {'trajectory': out["prediction"].squeeze(0).cpu().numpy()[:, :2], 'time': solve_time}

### 主实验循环 (Benchmark Loop)

In [9]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import time

# --- 0. 辅助类：适配 TOES 的分布接口 ---
class ProxyDistribution:
    """
    一个简单的包装器，把我们的 Tensor 热力图伪装成 TOES 需要的 Distribution 对象
    """
    def __init__(self, distribution_tensor):
        # distribution_tensor: [1, H, W] or [H, W]
        if distribution_tensor.dim() == 3:
            grid = distribution_tensor.squeeze(0)
        else:
            grid = distribution_tensor
        
        # TOES 需要 numpy 格式的 .evals
        self.evals = grid.cpu().numpy()

# --- 1. 初始化 ---
# 确保 toes_solver 和 diffusion_solver 已经由你之前的 Wrapper 类实例化
# 如果没有，请先运行定义这两个类的 Cell
if 'toes_solver' not in globals():
    toes_solver = TOESSolver()
if 'diffusion_solver' not in globals():
    diffusion_solver = DiffusionSolver(model, device)

# --- 2. 设置实验数据 ---
# 从验证集中取一个样本 (例如 Sample 0)
sample_idx = 0
sample = val_loader.dataset[sample_idx]

# 提取关键信息
# A. 轨迹起点和终点
# 注意：因为 Dataset 用 0 填充，我们需要找到最后一个有效点作为 Goal
gt_traj = sample["trajectories"].numpy()
valid_len = last_valid_idx_xy_np(gt_traj[:, :2])
start = gt_traj[0, :2]
end   = gt_traj[valid_len, :2]

print(f"Benchmark Sample {sample_idx}:")
print(f"  Start: {start}")
print(f"  Goal : {end}")

# B. 构建 TOES 需要的分布对象
dist_manager = ProxyDistribution(sample["distribution"])

# C. 准备 Diffusion 需要的输入 (转为 Tensor 并移动到 Device)
# 注意：这里需要保持 batch 维度 [1, ...] 以便 Solver 内部处理
diff_inputs = {
    "distribution_grid": sample["distribution"].to(device),
    "gaussian_packed": sample["gaussian_packed"].to(device),
    "gaussian_mask": sample["gaussian_padding_mask"].to(device)
}

# D. 准备度量工具 (用于统一计算 E)
# 确保使用与 TOES 相同的边界
METRIC_BOUNDS = ((0.0, 3.5), (-1.0, 3.5)) 
metric_eval = FourierErgodic(bounds=METRIC_BOUNDS, max_mode=6, lambda_power=1.25)
# 计算目标系数 (Ground Truth Coefficients)
dist_coeffs = metric_eval.compute_coefficients(sample["distribution"].to(device))

# --- 3. 主循环 ---
gamma_levels = [0.2, 0.1, 0.05, 0.01, 0.001]
results = {'toes': [], 'diffusion': []}

print(f"\n🚀 开始对比实验 (Gamma Levels: {gamma_levels})...")

for g in gamma_levels:
    print(f"\n--- Testing Gamma = {g} ---")
    
    # === A. 运行 TOES (Baseline) ===
    try:
        print("  Running TOES...")
        res_toes = toes_solver.solve(start, end, dist_manager, gamma=g)
        
        # 计算 E 值 (统一使用 metric_eval)
        # res_toes['trajectory'] 是 numpy [T, 2]，需转 tensor
        traj_toes = torch.tensor(res_toes['trajectory'], dtype=torch.float32, device=device).unsqueeze(0)
        e_toes = metric_eval.energy(traj_toes, dist_coeffs.unsqueeze(0)).item()
        
        results['toes'].append({
            'gamma': g, 
            'time': res_toes['time'], 
            'erg': e_toes,
            'traj': res_toes['trajectory'] # 保存轨迹以供画图
        })
        print(f"    [TOES] Time: {res_toes['time']:.4f}s | E: {e_toes:.6f}")
        
    except Exception as e:
        print(f"    [TOES] Failed: {e}")
        results['toes'].append({'gamma': g, 'time': np.nan, 'erg': np.nan, 'traj': None})

    # === B. 运行 Diffusion (Ours) ===
    try:
        print("  Running Diffusion...")
        # 调用 Solver 包装器
        res_diff = diffusion_solver.solve(
            start=start,
            distribution_grid=diff_inputs['distribution_grid'],
            gaussian_packed=diff_inputs['gaussian_packed'],
            gaussian_mask=diff_inputs['gaussian_mask'],
            gamma=g
        )
        
        # 计算 E 值
        traj_diff = torch.tensor(res_diff['trajectory'], dtype=torch.float32, device=device).unsqueeze(0)
        # 强制起点对齐以获得公平的 E
        traj_diff[:, 0, :] = torch.tensor(start, device=device)
        
        e_diff = metric_eval.energy(traj_diff, dist_coeffs.unsqueeze(0)).item()
        
        results['diffusion'].append({
            'gamma': g, 
            'time': res_diff['time'], 
            'erg': e_diff,
            'traj': res_diff['trajectory']
        })
        print(f"    [Diff] Time: {res_diff['time']:.4f}s | E: {e_diff:.6f}")
        
    except Exception as e:
        print(f"    [Diff] Failed: {e}")
        results['diffusion'].append({'gamma': g, 'time': np.nan, 'erg': np.nan, 'traj': None})

print("\n✅ 实验完成。")

# --- 4. 快速画图验证 ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 提取数据
gammas = gamma_levels
toes_time = [r['time'] for r in results['toes']]
diff_time = [r['time'] for r in results['diffusion']]
toes_erg = [r['erg'] for r in results['toes']]
diff_erg = [r['erg'] for r in results['diffusion']]

# 图 1: 计算时间对比 (Log Scale)
axes[0].plot(gammas, toes_time, 'b-o', label='TOES (Solver)')
axes[0].plot(gammas, diff_time, 'r-s', label='Diffusion (Ours)')
axes[0].set_xscale('log')
axes[0].set_yscale('log') # 时间通常差异巨大，用对数坐标
axes[0].set_xlabel(r'Gamma ($\gamma$)')
axes[0].set_ylabel('Computation Time (s)')
axes[0].invert_xaxis() # Gamma 越小越难，通常放在右边或左边看习惯
axes[0].set_title('Speed Comparison (Lower is Better)')
axes[0].legend()
axes[0].grid(True, which="both", ls="-", alpha=0.5)

# 图 2: 遍历性对比
axes[1].plot(gammas, toes_erg, 'b-o', label='TOES')
axes[1].plot(gammas, diff_erg, 'r-s', label='Diffusion')
axes[1].plot(gammas, gammas, 'k--', label='Target Gamma') # 参考线
axes[1].set_xscale('log')
axes[1].set_xlabel(r'Gamma ($\gamma$)')
axes[1].set_ylabel('Measured Ergodicity ($E$)')
axes[1].invert_xaxis()
axes[1].set_title('Quality Comparison (Lower is Better)')
axes[1].legend()
axes[1].grid(True)

# 图 3: Pareto 曲线 (Time vs Ergodicity)
axes[2].scatter(toes_time, toes_erg, c='b', label='TOES')
axes[2].scatter(diff_time, diff_erg, c='r', label='Diffusion')
axes[2].set_xlabel('Time (s)')
axes[2].set_ylabel('Ergodicity ($E$)')
axes[2].set_xscale('log')
axes[2].set_title('Pareto Frontier')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

NameError: name 'model' is not defined